# 00 Data check: do the five CIS files open and have the structure we expect?

**Goal:** confirm that the five Canadian Income Survey (CIS) files for 2018 to 2022 are in Google Drive, are fixed-width text, and have the number of person rows and households we report.

**Sources used for the code**
- Statistics Canada, *Canadian Income Survey, Public Use Microdata File* data dictionaries, 2018 to 2022 (Catalogue no. 72M0003X). Column positions below come from these.
- pandas documentation for `read_fwf`: https://pandas.pydata.org/docs/reference/api/pandas.read_fwf.html

**You should see at the end (rows = person records, households = unique PUMFID):**

| Year | Person rows | Households | FSCHHLDM same within household | One major income earner per household |
|---|---|---|---|---|
| 2018 | 94,336 | 40,857 | True | True |
| 2019 | 72,643 | 31,658 | True | True |
| 2020 | 123,718 | 53,882 | True | True |
| 2021 | 89,939 | 39,243 | True | True |
| 2022 | 96,302 | 41,978 | True | True |

## Step 1: Connect Google Drive to Colab

In [ ]:
# Import Colab's helper for connecting to Google Drive.
from google.colab import drive

# Mount (connect) your Drive so Colab can read the files stored there.
# Colab will ask you to sign in and allow access the first time.
drive.mount('/content/drive')

## Step 2: Check that the five files are where we expect

In [ ]:
# 'os' is Python's built-in tool for working with folders and files.
import os

# The folder in your Google Drive that holds the five CIS files.
# If you stored the files somewhere else, change this one line.
folder_path = '/content/drive/MyDrive/CIND820_CapstoneProject/ProjectData'

# Get the names of everything inside that folder.
files = os.listdir(folder_path)

# Print each file's full path, one per line.
print("Files in your directory:")
for f in files:
    full_path = os.path.join(folder_path, f)
    print(full_path)

# You should see five files: CIS2018_PUMF.txt ... CIS2022_PUMF.txt

## Step 3: Look at the first row of each file

The files are *fixed-width* text, not CSV. In a fixed-width file every row has the same number of characters, and each variable sits at a fixed position. A CSV would show commas, so we also count commas to confirm there are none.

In [ ]:
# pandas is the main Python library for working with tables of data.
import pandas as pd

# Repeat the same check for each of the five survey years.
for year in [2018, 2019, 2020, 2021, 2022]:
    # Open that year's file and read only its first line (row).
    with open(f'{folder_path}/CIS{year}_PUMF.txt') as f:
        # rstrip('\n') removes the invisible end-of-line character so it is not counted.
        line = f.readline().rstrip('\n')
    # Print: the year, the number of characters in the row, how many commas it has,
    # and its first 30 characters.
    print(year, 'length:', len(line), '| commas:', line.count(','), '| start:', line[:30])

# You should see: lengths 1188, 1188, 1189, 1854, 1890 and 0 commas for every year.
# The length differs after 2020 because the later files contain more variables.

## Step 4: Count person rows and households, and test two assumptions

Each row is one *person*, but food security (FSCHHLDM) is measured for the whole *household*. We test two things:
1. Everyone in the same household has the same FSCHHLDM value.
2. Each household has exactly one row flagged as the major income earner (HHMJIE = 1).

**Column positions.** The data dictionaries number character positions starting at 1 and give a start and end (for example HHMJIE 1120-1120 in 2019). pandas `read_fwf` counts from 0 and the end position is not included. So the dictionary start minus 1 gives the pandas start, and the dictionary end stays the same. Example: dictionary 1120-1120 becomes (1119, 1120).

In [ ]:
# Where each variable sits in each year's file, taken from the Statistics Canada data dictionaries.
# PUMFID   = household identifier
# HHMJIE   = flag: major income earner in the household (1 = yes)
# FSCHHLDM = household food security status (the outcome we want to predict)
pos = {
    2018: {'PUMFID': (4, 9), 'HHMJIE': (1119, 1120), 'FSCHHLDM': (1177, 1178)},
    2019: {'PUMFID': (4, 9), 'HHMJIE': (1119, 1120), 'FSCHHLDM': (1177, 1178)},
    2020: {'PUMFID': (4, 9), 'HHMJIE': (1119, 1120), 'FSCHHLDM': (1177, 1178)},
    2021: {'PUMFID': (4, 9), 'HHMJIE': (1784, 1785), 'FSCHHLDM': (1842, 1843)},
    2022: {'PUMFID': (4, 9), 'HHMJIE': (1820, 1821), 'FSCHHLDM': (1878, 1879)},
}

# Print a header so the output columns are easy to read.
print('year | person rows | households | FSCHHLDM same within household | one earner per household')

for year, cols in pos.items():
    # Read only the three columns we need, using their positions.
    # dtype=str keeps the values as text so codes like '01' are not changed.
    df = pd.read_fwf(f'{folder_path}/CIS{year}_PUMF.txt',
                     colspecs=list(cols.values()),
                     names=list(cols.keys()), dtype=str)

    # Turn the two code columns into whole numbers so we can compare and count them.
    df['HHMJIE'] = df['HHMJIE'].astype(int)
    df['FSCHHLDM'] = df['FSCHHLDM'].astype(int)

    # Test 1: for each household, count the different FSCHHLDM values it has.
    # If the largest count across all households is 1, everyone in a household agrees.
    same_fs = df.groupby('PUMFID')['FSCHHLDM'].nunique().max() == 1

    # Test 2: the number of rows flagged HHMJIE = 1 should equal the number of households.
    one_earner = (df['HHMJIE'] == 1).sum() == df['PUMFID'].nunique()

    # Print: year, person rows, unique households, then the two test results.
    print(year, len(df), df['PUMFID'].nunique(), same_fs, one_earner)

# Compare your output with the table at the top of this notebook.
# If a number differs, see docs/FAQ.md.